# Description

In this notebook, I will implement the scaled-dot product operation and compare the distance between
- torch built-in function
- custom function
- quantized function

In [25]:
import os 
import math
import time
import functools
import numpy as np 

import torch
from torch.nn.functional import scaled_dot_product_attention

In [26]:
def cuda_memory_profiler(device="cuda"):
    """
    Decorator that measures GPU memory usage (and runtime) for any function.
    
    Reports:
      - Δpeak (max temporary memory used)
      - Δcurrent (net memory retained after execution)
      - runtime (optional)
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            # synchronize before measuring
            torch.cuda.synchronize(device)
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
            
            before = torch.cuda.memory_allocated(device)

            result = func(*args, **kwargs)  # run the function

            torch.cuda.synchronize(device)
            peak = torch.cuda.max_memory_allocated(device)
            
            delta_peak = peak - before

            msg = (f"[{func.__name__}] Δpeak: {delta_peak/1e6:.2f} MB")
            print(msg)

            return result
        return wrapper
    return decorator

# 1. Torch built-in scaled do product

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dtype = torch.float16

Using device: cuda


In [28]:
SEQUENCE_LENGTH = 512 
D_MODEL = 128
# N_HEADS = 8

Q = torch.randn(SEQUENCE_LENGTH, D_MODEL, device=device, dtype=dtype)
K = torch.randn(SEQUENCE_LENGTH, D_MODEL, device=device, dtype=dtype)
V = torch.randn(SEQUENCE_LENGTH, D_MODEL, device=device, dtype=dtype)

# warmup
for _ in range(10):
    _ = scaled_dot_product_attention(Q, K, V, attn_mask=None, dropout_p=0.0, is_causal=False)

In [29]:
@cuda_memory_profiler()
def torch_built_in_scaled_dot_product(Q, K, V):
    z = scaled_dot_product_attention(Q, K, V)
    return z

z = torch_built_in_scaled_dot_product(Q, K, V)
print(f"Shape of output: {z.shape}")
print(f"Dtype of output: {z.dtype}")  

[torch_built_in_scaled_dot_product] Δpeak: 4.46 MB
Shape of output: torch.Size([512, 128])
Dtype of output: torch.float16


# 2. Normal (float) scaled-dot product

In [30]:
def custom_scaled_dot_product(Q, K, V):
    d_k = Q.size(-1)
    scale = 1 / math.sqrt(d_k)
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output

In [31]:
z_custom = custom_scaled_dot_product(Q, K, V)
print(f"Shape of custom output: {z_custom.shape}")
print(f"Dtype of custom output: {z_custom.dtype} \n")

if torch.allclose(z, z_custom, rtol=1e-2, atol=1e-2):
    print("Correct !! \n")
else:
    print("Custom implementation is WRONG !! \n")

Shape of custom output: torch.Size([512, 128])
Dtype of custom output: torch.float16 

Correct !! 



# 3. Int8 scaled-dot product (scalar scale)

In [32]:
def quantize_matrix_symmetric_int8(mat:torch.Tensor):
    """
    Symmetric quantization to int8.
    mat: input float matrix (e.g., torch.float32 or torch.bfloat16)
    """
    max_val = torch.max(torch.abs(mat))
    
    qmin = -128
    qmax = 127
    scale = max_val / qmax
    
    q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
    
    scale = scale.clone().detach().to(torch.float32)
    return q_mat, scale

def de_quantize_matrix_symmetric_int8(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 tensor to float.
    q_mat: input int8 tensor
    scale: scale factor (single value)
    """
    output = q_mat.to(torch.float32) 
    output = output * scale
    output = output.to(out_dtype)
    return output

def quantization_error_l2_norm(original, dequantized):
    """
    Compute the relative error between the original and dequantized tensors using l2 norm.
    """
    return torch.norm(original - dequantized)


def quantization_error_mse(original, dequantized):
    """
    Compute the Mean Squared Error (MSE) between the original and dequantized tensors.
    """
    return torch.mean((original - dequantized) ** 2)


def quantization_error_kl_divergence(original, dequantized, num_bins=2048, epsilon=1e-10):
    """
    Compute the KL divergence between the distributions of the original and dequantized tensors with float16.
    """
    orig_hist = torch.histc(original.float(), bins=num_bins, min=-1.0, max=1.0)
    deq_hist = torch.histc(dequantized.float(), bins=num_bins, min=-1.0, max=1.0)

    orig_prob = orig_hist / (torch.sum(orig_hist) + epsilon)
    deq_prob = deq_hist / (torch.sum(deq_hist) + epsilon)

    kl_div = torch.sum(orig_prob * torch.log((orig_prob + epsilon) / (deq_prob + epsilon)))
    return kl_div

In [36]:
def dummy_int8_matmul(A_int8:torch.Tensor, B_int:torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int8.dtype != torch.int8 or B_int.dtype != torch.int8:
        raise ValueError("Both A and B must be int8 tensors.")
    A_float = A_int8.to(torch.float16)
    B_float = B_int.to(torch.float16)
    result_float = torch.matmul(A_float, B_float)
    return result_float.to(out_dtype)

# def scaled_dot_product_int8(Q_q, Q_scale, K_q, K_scale, V):
#     dk = Q_q.size(-1)
#     scale = 1.0 / math.sqrt(dk)
    
#     scores_int32 = dummy_int8_matmul(Q_q, K_q.transpose(-2, -1))
    
#     print(f"score int32 dtype: {scores_int32.dtype}")
#     print(f"Q scale value: {Q_scale.item()}")
#     print(f"K scale value: {K_scale.item()}")
    
#     scores = scores_int32 * Q_scale * K_scale * scale
#     scores = scores.to(dtype)
    
#     attn_weights = torch.softmax(scores, dim=-1)
    
#     return torch.matmul(attn_weights, V)

def scaled_dot_product_int8(Q_q, Q_scale, K_q, K_scale, V_q, V_scale):
    dk = Q_q.size(-1)
    scale = 1.0 / math.sqrt(dk)
    
    scores_int32 = dummy_int8_matmul(Q_q, K_q.transpose(-2, -1))
    
    scores = scores_int32 * Q_scale * K_scale * scale
    scores = scores.to(dtype)
    
    attn_weights = torch.softmax(scores, dim=-1)
    attn_weight_q, attn_weight_scale = quantize_matrix_symmetric_int8(attn_weights)
    
    output_int = dummy_int8_matmul(attn_weight_q, V_q)
    output = output_int * attn_weight_scale * V_scale
    output = output.to(dtype)
    
    return output

In [37]:
# Quantize Q, K, V
Q_q, Q_scale = quantize_matrix_symmetric_int8(Q)
K_q, K_scale = quantize_matrix_symmetric_int8(K)
V_q, V_scale = quantize_matrix_symmetric_int8(V)

print(f"Q_q dtype: {Q_q.dtype} - scale: {Q_scale.dtype}")
print(f"K_q dtype: {K_q.dtype} - scale: {K_scale.dtype}")
print(f"V_q dtype: {V_q.dtype} - scale: {V_scale.dtype}")

Q_q dtype: torch.int8 - scale: torch.float32
K_q dtype: torch.int8 - scale: torch.float32
V_q dtype: torch.int8 - scale: torch.float32


In [38]:
z_int8 = scaled_dot_product_int8(Q_q, Q_scale, K_q, K_scale, V_q, V_scale)
print(f"Shape of int8 output: {z_int8.shape} - dtype: {z_int8.dtype}")

if torch.allclose(z, z_int8, rtol=1.0, atol=1.0):
    print("\n Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")

error_l2 = quantization_error_l2_norm(z, z_int8)
print(f"L2 norm difference between float and int8 outputs: {error_l2.item()}")

error_mse = quantization_error_mse(z, z_int8)
print(f"MSE between float and int8 outputs: {error_mse.item()}")

error_kl = quantization_error_kl_divergence(z, z_int8)
print(f"KL divergence between float and int8 outputs: {error_kl.item()}")

Shape of int8 output: torch.Size([512, 128]) - dtype: torch.float16

 Correct !! 

L2 norm difference between float and int8 outputs: 1.4423828125
MSE between float and int8 outputs: 3.17692756652832e-05
KL divergence between float and int8 outputs: 0.01304895430803299
